# RunVault — Interactive Quickstart Notebook

Welcome to **RunVault**! This notebook demonstrates how to:
1. Initialize experiments and track training runs using `runvault`.
2. Log hyperparameter configurations and stream scalar metrics.
3. **Save atomic, immutable checkpoints** with SHA-256 integrity validation.
4. **Restore checkpoints directly** using `CheckpointRecoveryResult` (`os.PathLike`).
5. **Automatic fault-tolerant fallback** when the latest checkpoint is corrupted.
6. Query persisted runs, parameters, metrics, and checkpoints from local SQLite.
7. Automatically capture and diagnose runtime failures.

## 1. Import RunVault and Initialize an Experiment

`runvault.init()` is the primary entrypoint. It creates or retrieves an experiment and manages the run lifecycle.

In [1]:
import math
import random
from pathlib import Path

import runvault

print(f"RunVault SDK version: {runvault.__version__}")

RunVault SDK version: 0.1.0


## 2. Execute a Baseline Training Run

Using `with runvault.init(...) as run:` guarantees runs are marked `FINISHED` on clean exit or `FAILED` on exception.

In [2]:
with runvault.init(
    experiment="cifar10-benchmark",
    name="baseline-sgd",
    tags={"optimizer": "sgd", "model": "resnet18"},
    description="Baseline training run on CIFAR-10 with standard SGD",
) as run:
    run.log_params(
        {
            "learning_rate": 0.01,
            "batch_size": 64,
            "momentum": 0.9,
            "weight_decay": 1e-4,
        }
    )

    print(f"Active Run ID: {run.id}")
    print(f"Status during execution: {run.status.value}")

    for epoch in range(5):
        train_loss = 1.8 / (1.0 + 0.3 * epoch) + random.uniform(-0.02, 0.02)
        accuracy = 0.45 + 0.45 * (1.0 - math.exp(-0.25 * epoch))

        run.log_metrics(
            {
                "train_loss": round(train_loss, 4),
                "accuracy": round(accuracy, 4),
            },
            step=epoch,
        )

print(f"Run ended. Final status: {run.status.value}")

Active Run ID: 96d21bee-1334-4f70-a10e-70f6bb190bbb
Status during execution: RUNNING
Run ended. Final status: FINISHED


## 3. Execute a Second Run with Atomic Checkpoints

RunVault's checkpoint engine saves model weights atomically using exclusive hard links (`os.link`) and calculates SHA-256 integrity checksums with `synchronous=FULL` SQLite durability.

In [3]:
with runvault.init(
    experiment="cifar10-benchmark",
    name="tuned-adamw",
    tags={"optimizer": "adamw", "model": "resnet18"},
) as run_tuned:
    run_tuned.log_params(
        {
            "learning_rate": 0.001,
            "batch_size": 128,
            "weight_decay": 0.01,
        }
    )

    for epoch in range(1, 7):
        train_loss = 1.5 / (1.0 + 0.45 * epoch) + random.uniform(-0.01, 0.01)
        accuracy = 0.52 + 0.42 * (1.0 - math.exp(-0.35 * epoch))

        run_tuned.log_metrics(
            {
                "train_loss": round(train_loss, 4),
                "accuracy": round(accuracy, 4),
            },
            step=epoch,
        )

        # Save atomic checkpoints at epochs 3 and 6
        if epoch in (3, 6):

            def save_weights(dest: Path, ep: int = epoch, acc: float = accuracy) -> None:
                # In real training: torch.save(model.state_dict(), dest)
                dest.write_text(f"MODEL_STATE_EPOCH_{ep}_ACC_{acc:.4f}\n")

            ckpt = run_tuned.save_checkpoint(
                name=f"checkpoint-epoch-{epoch}",
                writer=save_weights,
                step=epoch,
                metadata={"epoch": epoch, "accuracy": round(accuracy, 4)},
            )
            print(f"Saved Checkpoint '{ckpt.name}' (Step {ckpt.step}, Commit #{ckpt.commit_seq})")

print(f"Tuned run ended. Final status: {run_tuned.status.value}")

Saved Checkpoint 'checkpoint-epoch-3' (Step 3, Commit #5)
Saved Checkpoint 'checkpoint-epoch-6' (Step 6, Commit #6)
Tuned run ended. Final status: FINISHED


## 4. Restore Checkpoint for Training Recovery

The returned `CheckpointRecoveryResult` implements `os.PathLike[str]` directly and includes convenience methods such as `.read_bytes()` and `.read_text()`.

In [4]:
# CheckpointRecoveryResult implements os.PathLike[str]
recovery = run_tuned.restore_checkpoint()

print(f"Restored Checkpoint:  {recovery.checkpoint.name}")
print(f"Commit Sequence:      #{recovery.checkpoint.commit_seq}")
print(f"Artifact Path:        {recovery.path.name}")
print(f"Payload:              {recovery.read_text().strip()}")
print(f"Is Fallback Recovery: {recovery.is_fallback}")

Restored Checkpoint:  checkpoint-epoch-6
Commit Sequence:      #6
Artifact Path:        e4236064-eee8-46d5-9915-4fc8825fde41.bin
Payload:              MODEL_STATE_EPOCH_6_ACC_0.8886
Is Fallback Recovery: False


## 5. Fault-Tolerant Checkpoint Recovery (Automatic Fallback)

If the latest checkpoint was corrupted or interrupted during training, RunVault automatically detects the integrity failure and falls back to the latest valid older checkpoint.

In [5]:
# Deliberately corrupt the latest checkpoint on disk to simulate incomplete write / bit-rot
latest_ckpt_file = recovery.path
original_bytes = latest_ckpt_file.read_bytes()
latest_ckpt_file.write_bytes(b"CORRUPTED_BYTES")

print(f"Corrupted latest checkpoint on disk: {latest_ckpt_file.name}")
print("Attempting automatic checkpoint recovery...")

# RunVault detects the hash mismatch and transparently falls back to the previous intact checkpoint!
fallback_recovery = runvault.restore_checkpoint(run_tuned.id)

print(f"\nFell back to Checkpoint: {fallback_recovery.checkpoint.name}")
print(f"Commit Sequence:         #{fallback_recovery.checkpoint.commit_seq}")
print(f"Payload Content:         {fallback_recovery.read_text().strip()}")
print(f"Is Fallback:             {fallback_recovery.is_fallback}")
print(f"Rejected Candidates:     {len(fallback_recovery.rejected_checkpoints)}")
for rejected_ckpt, reason in fallback_recovery.rejected_checkpoints:
    print(f"  - {rejected_ckpt.name}: {reason}")

# Restore original checkpoint bytes
latest_ckpt_file.write_bytes(original_bytes)

Corrupted latest checkpoint on disk: e4236064-eee8-46d5-9915-4fc8825fde41.bin
Attempting automatic checkpoint recovery...

Fell back to Checkpoint: checkpoint-epoch-3
Commit Sequence:         #5
Payload Content:         MODEL_STATE_EPOCH_3_ACC_0.7930
Is Fallback:             True
Rejected Candidates:     1
  - checkpoint-epoch-6: Artifact size mismatch: expected 31 bytes, got 15 bytes


31

## 6. Query Previous Experiments, Runs & Checkpoints from SQLite

Query all historical runs, parameters, metrics, and committed checkpoints directly from the local SQLite database.

In [6]:
exp = runvault.get_experiment_by_name("cifar10-benchmark")
assert exp is not None

print(f"Experiment: {exp.name}")
print(f"Created:    {exp.created_at.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("-" * 65)

for r in exp.list_runs():
    params = runvault.get_params(r.id)
    ckpts = runvault.list_checkpoints(r.id)
    dur = f"{r.duration_seconds:.2f}s" if r.duration_seconds is not None else "N/A"
    print(
        f"Run: {r.name:<15} | Status: {r.status.value:<9} | Checkpoints: {len(ckpts)} | Dur: {dur}"
    )

print(f"\nCheckpoints for '{run_tuned.name}':")
for c in runvault.list_checkpoints(run_tuned.id):
    print(
        f"  Commit #{c.commit_seq}: {c.name:<20} | "
        f"Step {c.step} | Size: {c.size_bytes}B | SHA: {c.checksum_sha256[:10]}..."
    )

Experiment: cifar10-benchmark
Created:    2026-11-15 20:13:46 UTC
-----------------------------------------------------------------
Run: tuned-adamw     | Status: FINISHED  | Checkpoints: 2 | Dur: 0.19s
Run: baseline-sgd    | Status: FINISHED  | Checkpoints: 0 | Dur: 0.01s
Run: faulty-run      | Status: FAILED    | Checkpoints: 0 | Dur: 0.01s
Run: tuned-adamw     | Status: FINISHED  | Checkpoints: 2 | Dur: 0.08s
Run: baseline-sgd    | Status: FINISHED  | Checkpoints: 0 | Dur: 0.01s
Run: faulty-run      | Status: FAILED    | Checkpoints: 0 | Dur: 0.00s
Run: tuned-adamw     | Status: FINISHED  | Checkpoints: 2 | Dur: 0.06s
Run: baseline-sgd    | Status: FINISHED  | Checkpoints: 0 | Dur: 0.01s
Run: faulty-run      | Status: FAILED    | Checkpoints: 0 | Dur: 0.00s
Run: tuned-adamw     | Status: FINISHED  | Checkpoints: 0 | Dur: 0.00s
Run: baseline-sgd    | Status: FINISHED  | Checkpoints: 0 | Dur: 0.01s

Checkpoints for 'tuned-adamw':
  Commit #5: checkpoint-epoch-3   | Step 3 | Size: 31B 

## 7. Automatic Failure Tracking

If an unhandled exception aborts a training run, RunVault records the error type, message, and traceback in SQLite and sets the run status to `FAILED`.

In [7]:
try:
    with runvault.init("cifar10-benchmark", name="faulty-run") as faulty_run:
        faulty_run.log_param("stage", "data_loading")
        # Simulate a crash
        raise RuntimeError("CUDA out of memory (simulated error)")
except RuntimeError as e:
    print(f"Caught expected exception: {e}")

# Verify that the run recorded the failure in SQLite
failed_record = runvault.get_run(faulty_run.id)
assert failed_record is not None
print(f"Run Status:     {failed_record.status.value}")
print(f"Error Type:     {failed_record.error_type}")
print(f"Error Message:  {failed_record.error_message}")

Caught expected exception: CUDA out of memory (simulated error)
Run Status:     FAILED
Error Type:     RuntimeError
Error Message:  CUDA out of memory (simulated error)


## 8. Remote Checkpoint Storage with Google Drive (`runvault[gdrive]`)

RunVault supports **direct-to-storage** remote checkpointing with Google Drive.
Serialized model weights stream directly between your training runtime and Google Drive APIs, bypassing any metadata or proxy servers.

### Key Capabilities & Invariants
- **Zero-Configuration in Google Colab:** Automatically authenticates via `google.colab.auth`.
- **Secure Local OAuth:** Restrictive `0o600` token permissions inside `0o700` directories with account isolation.
- **Resumable Upload Idempotency:** Automatic recovery of completed uploads on network drop or lost HTTP responses.
- **Two-Phase Concurrency Collision Prevention:** Preflight check and post-upload verification prevent duplicate artifact creation across distributed workers.
- **Bounded-Memory Streaming:** Checkpoint downloads consume constant $O(\text{chunk\_size})$ RAM without whole-file buffering using `with recovery.open() as stream:`.

In [8]:
from runvault.integrations.gdrive import (
    check_gdrive_dependencies,
    is_colab_environment,
)

# Verify optional Google Drive SDK dependencies
check_gdrive_dependencies()
print("Google Drive SDK dependencies available: True")
print(f"Executing in Google Colab environment:    {is_colab_environment()}")

Google Drive SDK dependencies available: True
Executing in Google Colab environment:    False


### Live Usage in Google Colab and Local Desktop

In **Google Colab**, simply instantiate `GoogleDriveStorageProvider()` and pass it to `runvault.init(storage_provider=...)`:

```python
import torch
from runvault.integrations.gdrive import GoogleDriveStorageProvider

# Auto-authenticates via Colab runtime
gdrive = GoogleDriveStorageProvider(root_folder_name="RunVaultCheckpoints")

with runvault.init("colab-experiment", storage_provider=gdrive) as run:
    run.save_checkpoint(
        name="epoch-10",
        writer=lambda dest: torch.save(model.state_dict(), dest),
        step=10,
    )
    
    # Restore checkpoint directly from Drive with streaming verification
    recovery = run.restore_checkpoint()
    with recovery.open() as stream:
        state_dict = torch.load(stream)
```

On a **local desktop**, authenticate via browser OAuth with secure token caching:
```python
gdrive = GoogleDriveStorageProvider.from_oauth(
    client_secrets_file="client_secret.json",
    token_cache_path="~/.config/runvault/gdrive_token.json",
)
```

### Interactive End-to-End Demonstration (Offline-Safe & Self-Contained)

The cell below runs a complete, self-contained end-to-end demonstration of the Google Drive storage lifecycle using an in-memory Drive API mock. It validates direct-to-storage artifact publication, metadata tracking, SHA-256 verification, and bounded-memory streaming.

In [9]:
from pathlib import Path

from tests.test_gdrive import FakeDriveService

import runvault
from runvault.integrations.gdrive import GoogleDriveStorageProvider

# Initialize an in-memory Google Drive service for safe offline interactive testing
fake_drive = FakeDriveService()
gdrive_storage = GoogleDriveStorageProvider(drive_service=fake_drive)

print("Created GoogleDriveStorageProvider with Google Drive backend.")

# Run training with direct-to-Drive checkpoint persistence
with runvault.init(
    experiment="cifar10-benchmark",
    name="gdrive-remote-run",
    tags={"storage": "gdrive", "runtime": "cloud"},
    storage_provider=gdrive_storage,
) as gdrive_run:
    gdrive_run.log_params({"batch_size": 256, "remote_storage": "google_drive"})

    # Save checkpoint directly to Google Drive
    ckpt = gdrive_run.save_checkpoint(
        name="remote-checkpoint-final",
        writer=lambda dest: dest.write_bytes(b"GDRIVE_SERIALIZED_MODEL_WEIGHTS_VERSION_4_2_1\n"),
        step=100,
        metadata={"val_acc": 0.954, "storage_provider": "gdrive"},
    )
    print("Successfully uploaded checkpoint to Google Drive!")
    print(f"  Checkpoint Name:   {ckpt.name}")
    print(f"  Drive File ID:     {ckpt.storage_key}")
    print(f"  Storage Provider:  {ckpt.storage_provider}")
    print(f"  SHA-256 Checksum:  {ckpt.checksum_sha256[:16]}...")

# Restore checkpoint from Google Drive using bounded-memory streaming
recovery = runvault.restore_checkpoint(gdrive_run.id, storage=gdrive_storage)
print("\nRestored Checkpoint from Google Drive:")
print(f"  Checkpoint:        {recovery.checkpoint.name}")
print(f"  Storage Provider:  {recovery.checkpoint.storage_provider}")
print(f"  Drive File ID:     {recovery.checkpoint.storage_key}")
print(f"  Direct Path:       {recovery.path} (None for remote streaming backends)")

# Stream checkpoint payload in bounded memory
with recovery.open() as stream:
    content = stream.read()
    print(f"  Streamed Payload:  {content.decode('utf-8').strip()}")

Created GoogleDriveStorageProvider with Google Drive backend.
Successfully uploaded checkpoint to Google Drive!
  Checkpoint Name:   remote-checkpoint-final
  Drive File ID:     gdrive_file_3
  Storage Provider:  gdrive
  SHA-256 Checksum:  1ea33ee034a60f07...

Restored Checkpoint from Google Drive:
  Checkpoint:        remote-checkpoint-final
  Storage Provider:  gdrive
  Drive File ID:     gdrive_file_3
  Direct Path:       None (None for remote streaming backends)
  Streamed Payload:  GDRIVE_SERIALIZED_MODEL_WEIGHTS_VERSION_4_2_1
